# Generate Unaligned Resampling Test Data

Creates load profiles where the resample window does NOT cover all input
entries. Energy is intentionally NOT conserved because entries outside the
resample window are excluded.

Saved to `cases/`.

In [1]:
import datetime
import os

from ethos_penalps.post_processing.load_profiles.load_profile_entry_post_processor import (
    LoadProfileEntryPostProcessor,
)
from ethos_penalps.testing.load_profile.load_profil_creator import make_load_profile_entries_from_tuples
from ethos_penalps.testing.load_profile.resampling_helpers import save_resampling_test_case

## Output directory and helper

In [ ]:
CASES_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "cases")


def generate_unaligned_case(
    case_name: str,
    tuples: list,
    entry_start: datetime.datetime,
    resample_start: datetime.datetime,
    resample_end: datetime.datetime,
    expected_total_energy: float,
    resample_frequency: str = "1min",
):
    """Generate a case where the resample window differs from the entry range."""
    entries = make_load_profile_entries_from_tuples(entry_start, tuples)
    entry_end = entry_start + sum((d for _, d in tuples), datetime.timedelta())

    # Build meta with full entry range
    processor = LoadProfileEntryPostProcessor()
    meta = processor.create_load_profile_meta_data(
        list_of_load_profile_entries=entries,
        start_date_time_series=entry_start,
        end_date_time_series=entry_end,
        object_name="TestObject",
        object_type="TestType",
    )

    # Resample with different (unaligned) window
    result = processor.resample_load_profile_meta_data(
        load_profile_meta_data=meta,
        start_date=resample_start,
        end_date=resample_end,
        resample_frequency=resample_frequency,
    )

    parameters = {
        "entry_start": entry_start.isoformat(),
        "entry_end": entry_end.isoformat(),
        "start": resample_start.isoformat(),
        "end": resample_end.isoformat(),
        "resample_frequency": resample_frequency,
        "expected_total_energy": expected_total_energy,
    }

    save_resampling_test_case(
        meta=meta,
        result=result,
        cases_dir=CASES_DIR,
        parameters=parameters,
        case_name=case_name,
    )

    input_energy = meta.total_energy
    output_energy = result.total_energy
    print(f"Input energy:     {input_energy:.6f} {meta.energy_unit}")
    print(f"Resampled energy: {output_energy:.6f} {result.energy_unit}")
    print(f"Expected energy:  {expected_total_energy:.6f} {meta.energy_unit}")
    print(f"Energy lost:      {input_energy - output_energy:.6f} {meta.energy_unit}")
    print()

## Case 1 — Start trimmed

5 entries of 1 min each: [10, 20, 30, 40, 50] MJ.
Resample starts at minute 2, so the first 2 entries (10 + 20 = 30 MJ) are lost.
Expected output: bins with [30, 40, 50] MJ, total = 120 MJ, lost = 30 MJ.

In [ ]:
START = datetime.datetime(2021, 1, 1)
ONE_MIN = datetime.timedelta(minutes=1)

generate_unaligned_case(
    case_name="start_trimmed",
    tuples=[
        (10.0, ONE_MIN),
        (20.0, ONE_MIN),
        (30.0, ONE_MIN),
        (40.0, ONE_MIN),
        (50.0, ONE_MIN),
    ],
    entry_start=START,
    resample_start=START + datetime.timedelta(minutes=2),
    resample_end=START + datetime.timedelta(minutes=5),
    expected_total_energy=120.0,
    resample_frequency="1min",
)

## Case 2 — End trimmed

5 entries of 1 min each: [10, 20, 30, 40, 50] MJ.
Resample ends at minute 3, so the last 2 entries (40 + 50 = 90 MJ) are lost.
Expected output: bins with [10, 20, 30] MJ, total = 60 MJ, lost = 90 MJ.

In [ ]:
generate_unaligned_case(
    case_name="end_trimmed",
    tuples=[
        (10.0, ONE_MIN),
        (20.0, ONE_MIN),
        (30.0, ONE_MIN),
        (40.0, ONE_MIN),
        (50.0, ONE_MIN),
    ],
    entry_start=START,
    resample_start=START,
    resample_end=START + datetime.timedelta(minutes=3),
    expected_total_energy=60.0,
    resample_frequency="1min",
)

## Case 3 — Both sides trimmed

5 entries of 1 min each: [10, 20, 30, 40, 50] MJ.
Resample window is minute 1–4, so first entry (10 MJ) and last entry (50 MJ) are lost.
Expected output: bins with [20, 30, 40] MJ, total = 90 MJ, lost = 60 MJ.

In [ ]:
generate_unaligned_case(
    case_name="both_sides_trimmed",
    tuples=[
        (10.0, ONE_MIN),
        (20.0, ONE_MIN),
        (30.0, ONE_MIN),
        (40.0, ONE_MIN),
        (50.0, ONE_MIN),
    ],
    entry_start=START,
    resample_start=START + datetime.timedelta(minutes=1),
    resample_end=START + datetime.timedelta(minutes=4),
    expected_total_energy=90.0,
    resample_frequency="1min",
)

## Case 4 — Mid-entry trim with 2-minute entries

3 entries of 2 min each: [60, 120, 180] MJ (= 30, 60, 90 MJ/min).
Resample window is minute 1–5, cutting through the first and last entries.
First entry: 30 MJ/min × 1 min inside window = 30 MJ kept, 30 MJ lost.
Last entry: 90 MJ/min × 1 min inside window = 90 MJ kept, 90 MJ lost.
Expected output: 4 bins, total = 30 + 120 + 90 = 240 MJ, lost = 120 MJ.

In [ ]:
TWO_MIN = datetime.timedelta(minutes=2)

generate_unaligned_case(
    case_name="mid_entry_trim_varying",
    tuples=[
        (60.0, TWO_MIN),
        (120.0, TWO_MIN),
        (180.0, TWO_MIN),
    ],
    entry_start=START,
    resample_start=START + datetime.timedelta(minutes=1),
    resample_end=START + datetime.timedelta(minutes=5),
    expected_total_energy=240.0,
    resample_frequency="1min",
)

---

## Single-bin overlap cases

All cases below use a 10-minute resample window [05:00, 05:10) with
`resample_frequency="10min"` producing exactly 1 output bin. They
systematically cover every overlap type between an input entry and
the resample window.

In [ ]:
T = datetime.datetime(2022, 1, 1, 5, 0, 0)  # resample window start
WINDOW = datetime.timedelta(minutes=10)
M = datetime.timedelta(minutes=1)

## Case 5 — Left overlap (single bin)

300 MJ over 10 min, starting 5 min before window.
Window sees last 5 min -> 150 MJ.

In [ ]:
generate_unaligned_case(
    case_name="single_bin_left_overlap",
    tuples=[(300.0, 10 * M)],
    entry_start=T - 5 * M,
    resample_start=T,
    resample_end=T + WINDOW,
    expected_total_energy=150.0,
    resample_frequency="10min",
)

## Case 6 — Right overlap (single bin)

400 MJ over 10 min, starting 5 min into window.
Window sees first 5 min -> 200 MJ.

In [ ]:
generate_unaligned_case(
    case_name="single_bin_right_overlap",
    tuples=[(400.0, 10 * M)],
    entry_start=T + 5 * M,
    resample_start=T,
    resample_end=T + WINDOW,
    expected_total_energy=200.0,
    resample_frequency="10min",
)

## Case 7 — Fully enclosed (single bin)

500 MJ over 5 min, starting 2.5 min into window.
Fully inside -> 500 MJ.

In [ ]:
generate_unaligned_case(
    case_name="single_bin_fully_enclosed",
    tuples=[(500.0, 5 * M)],
    entry_start=T + datetime.timedelta(minutes=2, seconds=30),
    resample_start=T,
    resample_end=T + WINDOW,
    expected_total_energy=500.0,
    resample_frequency="10min",
)

## Case 8 — Exact match (single bin)

500 MJ over 10 min, exactly matching the window.
All energy included -> 500 MJ.

In [ ]:
generate_unaligned_case(
    case_name="single_bin_exact_match",
    tuples=[(500.0, 10 * M)],
    entry_start=T,
    resample_start=T,
    resample_end=T + WINDOW,
    expected_total_energy=500.0,
    resample_frequency="10min",
)

## Case 9 — Both sides overlap (single bin)

75 MJ over 15 min = 5 MJ/min, starting 2.5 min before window.
Window sees 10 min -> 50 MJ.

In [ ]:
generate_unaligned_case(
    case_name="single_bin_both_sides_overlap",
    tuples=[(75.0, 15 * M)],
    entry_start=T - datetime.timedelta(minutes=2, seconds=30),
    resample_start=T,
    resample_end=T + WINDOW,
    expected_total_energy=50.0,
    resample_frequency="10min",
)

## Case 10 — Completely before window (single bin)

120 MJ over 5 min, ending 5 min before window starts.
No overlap -> 0 MJ.

In [ ]:
generate_unaligned_case(
    case_name="single_bin_before_window",
    tuples=[(120.0, 5 * M)],
    entry_start=T - 10 * M,
    resample_start=T,
    resample_end=T + WINDOW,
    expected_total_energy=0.0,
    resample_frequency="10min",
)

## Case 11 — Completely after window (single bin)

170 MJ over 4 min, starting 30 min after window.
No overlap -> 0 MJ.

In [ ]:
generate_unaligned_case(
    case_name="single_bin_after_window",
    tuples=[(170.0, 4 * M)],
    entry_start=T + 30 * M,
    resample_start=T,
    resample_end=T + WINDOW,
    expected_total_energy=0.0,
    resample_frequency="10min",
)

## Case 12 — Mixed overlaps (single bin)

3 entries of 5 min each: [5, 10, 40] MJ, starting 2.5 min before window.
- Entry 1 [-2.5, +2.5]: left overlap, 2.5 min inside -> 2.5 MJ
- Entry 2 [+2.5, +7.5]: fully enclosed -> 10 MJ
- Entry 3 [+7.5, +12.5]: right overlap, 2.5 min inside -> 20 MJ
Total: 32.5 MJ

In [ ]:
generate_unaligned_case(
    case_name="single_bin_mixed_overlaps",
    tuples=[(5.0, 5 * M), (10.0, 5 * M), (40.0, 5 * M)],
    entry_start=T - datetime.timedelta(minutes=2, seconds=30),
    resample_start=T,
    resample_end=T + WINDOW,
    expected_total_energy=32.5,
    resample_frequency="10min",
)